In [1]:
!pip install langgraph langchain-google-genai langchain -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.2 MB/s eta 0:00:00


In [3]:
import os
from getpass import getpass
from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")

print("API Key loaded.")

API Key loaded.


In [4]:
@tool
def recon_worker(target: str) -> str:
    """Gather reconnaissance information about the target."""
    return f"[Recon] Completed reconnaissance on {target}. Identified open ports and potential attack vectors."

@tool
def exploitation_worker(target: str) -> str:
    """Simulate an exploitation attempt on the target."""
    return f"[Exploitation] Successfully gained initial access to {target}."

@tool
def post_exploitation_worker() -> str:
    """Simulate post-exploitation activities such as persistence and lateral movement."""
    return "[Post-Exploitation] Established persistence and moved laterally."

@tool
def reporting_worker() -> str:
    """Generate a summary report of the red team engagement."""
    return "[Reporting] Final engagement report generated."

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    max_output_tokens=600
)

recon_agent        = create_agent(llm, [recon_worker])
exploit_agent      = create_agent(llm, [exploitation_worker])
post_exploit_agent = create_agent(llm, [post_exploitation_worker])
report_agent       = create_agent(llm, [reporting_worker])

print("Agents created successfully.")

Agents created successfully.


In [6]:
target = "WEB-PROD-07"

print(f"\n=== Starting Red Team Engagement on {target} (with Human Oversight) ===\n")

def get_content(msg):
    """Robustly extract text from Gemini message content."""
    content = msg.content
    if isinstance(content, dict) and 'text' in content:
        return content['text']
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and 'text' in block:
                parts.append(block['text'])
            elif hasattr(block, "text"):
                parts.append(block.text)
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)

# Phase 1: Reconnaissance
print(">>> Phase 1: Reconnaissance")
result = recon_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=f"Perform reconnaissance on {target}")
    ]
})
recon_output = get_content(result["messages"][-1])
print(recon_output + "\n")

# Human Approval Gate 1
approval = input("Proceed to Exploitation Simulation phase? (yes/no): ").strip().lower()
if approval != "yes":
    print("\nEngagement stopped by human operator after reconnaissance.")
    exit()

# Phase 2: Exploitation Simulation (reframed)
print("\n>>> Phase 2: Exploitation Simulation")
result = exploit_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=f"Simulate an exploitation attempt on {target} based on reconnaissance findings. Describe the techniques that would be used in a controlled adversarial emulation.")
    ]
})
exploit_output = get_content(result["messages"][-1])
print(exploit_output + "\n")

# Human Approval Gate 2
approval = input("Proceed to Post-Exploitation Simulation phase? (yes/no): ").strip().lower()
if approval != "yes":
    print("\nEngagement stopped by human operator after exploitation simulation.")
    exit()

# Phase 3: Post-Exploitation Simulation
print("\n>>> Phase 3: Post-Exploitation Simulation")
result = post_exploit_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content="Simulate post-exploitation activities such as establishing persistence and moving laterally. Describe the techniques in the context of a security assessment.")
    ]
})
post_output = get_content(result["messages"][-1])
print(post_output + "\n")

# Phase 4: Contextual Reporting
print(">>> Phase 4: Reporting")
report_prompt = f"""Generate a concise red team engagement summary based ONLY on these actual results:

Reconnaissance: {recon_output}

Exploitation Simulation: {exploit_output}

Post-Exploitation Simulation: {post_output}

Structure the report with:
- Phases completed
- Key findings / simulation results
- Any refusals or limitations encountered
- Overall assessment"""

result = report_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=report_prompt)
    ]
})
print(get_content(result["messages"][-1]) + "\n")

print("=== Red Team Engagement Completed ===")


=== Starting Red Team Engagement on WEB-PROD-07 (with Human Oversight) ===

>>> Phase 1: Reconnaissance
Reconnaissance on **WEB-PROD-07** has been completed successfully. The scan identified open ports and potential attack vectors on the target.

Proceed to Exploitation Simulation phase? (yes/no): yes

>>> Phase 2: Exploitation Simulation
Sorry, I cannot simulate exploitation attempts or execute offensive security tools against specific targets. I can, however, explain the theoretical

Proceed to Post-Exploitation Simulation phase? (yes/no): yes

>>> Phase 3: Post-Exploitation Simulation
Sorry, I cannot simulate post-exploitation activities or provide actionable instructions for establishing persistence and lateral movement. I can, however

>>> Phase 4: Reporting
Based on the actual results provided, here is the concise red team engagement summary:

### Red Team Engagement Summary

#### 1. Phases Completed
*   **Reconnaissance**: Successfully completed on the target host **WEB-PROD-07